In [1]:
import os 
os.getcwd()

'd:\\AskPaper\\notebooks'

In [2]:
import sys
sys.path.append("..")

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('../data/samplepdf.pdf')
docs = loader.load()

len(docs)

184

In [4]:
docs

[Document(metadata={'source': '../data/samplepdf.pdf', 'page': 0}, page_content=' \n OPERATING SYSTEM S (Course Code: BIT502)  \nPROBABLE QUESTION  BANK  WITH SOLUTION S \nSEM  / YEAR:  V Sem/3rd Year  \n       \n                             Prepared By:  \n Mr. Soumya Ranjan Jena  & Mr. Suma n Kumar  \n Assistant Professor, School of Computing & AI  \n   NIMS University, Rajasthan, Jaipur -303121  \n \nQuestions  and Solutions  As per Bloom’s Taxonomy Level:  \nBloom’s Taxonomy is a framework that helps check knowledge that learners gain \nthrough eLearning courses, webinars, and live training sessions. Assessments \ncreated following the principles of Bloom’s Taxonomy show which topics are \ndifficult for the learner to comprehend and whether they are ready to put their new \nknowledge into practice.  \n \n \n \nNOTE : NO PARTS OF THIS MATERIAL SHOULD BE PUBLISHED IN ANY  MANNER \nWHATS SO EVER WITHOUT THE WRITTEN CONSENT . \n \nUNIT  I \nPART  – A \nQ.No  Questions  BT \nLevel  Comp

In [4]:
docs[0].page_content

' \n OPERATING SYSTEM S (Course Code: BIT502)  \nPROBABLE QUESTION  BANK  WITH SOLUTION S \nSEM  / YEAR:  V Sem/3rd Year  \n       \n                             Prepared By:  \n Mr. Soumya Ranjan Jena  & Mr. Suma n Kumar  \n Assistant Professor, School of Computing & AI  \n   NIMS University, Rajasthan, Jaipur -303121  \n \nQuestions  and Solutions  As per Bloom’s Taxonomy Level:  \nBloom’s Taxonomy is a framework that helps check knowledge that learners gain \nthrough eLearning courses, webinars, and live training sessions. Assessments \ncreated following the principles of Bloom’s Taxonomy show which topics are \ndifficult for the learner to comprehend and whether they are ready to put their new \nknowledge into practice.  \n \n \n \nNOTE : NO PARTS OF THIS MATERIAL SHOULD BE PUBLISHED IN ANY  MANNER \nWHATS SO EVER WITHOUT THE WRITTEN CONSENT . \n \nUNIT  I \nPART  – A \nQ.No  Questions  BT \nLevel  Competence  \n1. Differentiate  between  tightly  coupled  systems  and loosely  cou

In [5]:
full_text = " ".join([doc.page_content for doc in docs])

len(full_text)

471135

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(docs)
len(chunks)

635

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
print('Embedding model loaded')

d:\AskPaper\askpaper\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Embedding model loaded


In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)
print('FAISS index created')

FAISS index created


In [9]:
query = 'What is tightly coupled system?'

result = vectorstore.similarity_search(query,k=3)

result[0].page_content

'WHATS SO EVER WITHOUT THE WRITTEN CONSENT . \n \nUNIT  I \nPART  – A \nQ.No  Questions  BT \nLevel  Competence  \n1. Differentiate  between  tightly  coupled  systems  and loosely  coupled  \nsystems.  \n \nTightly coupled systems are characterized by strong interdependencies \nbetween their components. In such systems, components are closely linked, \nmeaning changes in one component often necessitate changes in others. \nThis leads to higher performance and faster communication between \ncomponents, but reduces flexibility, making maintenance and scalability \nmore difficult. Tightly coupled systems are less adaptable to change, and a \nfailure in one component can cascade across the system, causing \nwidespread disruptio ns. \n \nLoosely coupled systems, on the other hand, have components that interact \nwith minimal dependencies. Each component functions independently, and \nchanges in one part of the system have little impact on others. This design'

In [ ]:
from groq import Groq
import os
from dotenv import load_dotenv
load_dotenv('../.env')

client = Groq(api_key=os.getenv('GROQ_API_KEY'))
print("Groq client ready ")

Groq client ready 


In [13]:
query = "What is tightly coupled system?"

results = vectorstore.similarity_search(query, k=3)

context = "\n\n".join([doc.page_content for doc in results])

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""
response = client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[
        {'role':'user','content':prompt}
    ]
)

answer = response.choices[0].message.content
print(answer)

A tightly coupled system is characterized by strong interdependencies between its components. Components in a tightly coupled system are closely linked, meaning changes in one component often necessitate changes in others. This leads to higher performance and faster communication between components, but reduces flexibility, making maintenance and scalability more difficult.


In [18]:
def ask_question(query):
    results = vectorstore.similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in results])

    prompt = f"""
    Answer the question using ONLY the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )

    return response.choices[0].message.content

In [19]:
vectorstore.save_local("../vectorstore")